# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahadumar/flyrank-MLinternship-Assignment-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

# Load token from Colab secrets
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
print('Token loaded: ' + HF_TOKEN[:8] + '...')

con = duckdb.connect()
con.execute("CREATE SECRET hf_secret (TYPE huggingface, TOKEN '" + HF_TOKEN + "')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = REL + '/fact_content_daily_performance/**/*.parquet'

DEV_MONTH    = '2026-03'
PRIOR_MONTH  = '2026-02'
SEALED_MONTH = '2026-06'

print('Development month : ' + DEV_MONTH)
print('Prior month       : ' + PRIOR_MONTH)
print('Sealed test month : ' + SEALED_MONTH + '  <- never touched')
print('\nSetup complete. Ready.')

Token loaded: hf_ePFOh...
Development month : 2026-03
Prior month       : 2026-02
Sealed test month : 2026-06  <- never touched

Setup complete. Ready.


In [ ]:
# Rebuild the feature frame (same logic as w03)
print('Building feature frame...')

current = con.sql("""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)      AS monthly_impressions,
        SUM(gsc_clicks)           AS monthly_clicks,
        AVG(gsc_avg_position)     AS avg_position,
        SUM(ga4_engaged_sessions) AS monthly_engaged_sessions
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet',
                      hive_partitioning=true)
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

prior = con.sql("""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS prior_impressions
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet',
                      hive_partitioning=true)
    WHERE month = '2026-02'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feat = current.merge(prior, on='content_hash_id', how='inner')

# Safe CTR
feat['ctr'] = (
    feat['monthly_clicks'] /
    feat['monthly_impressions'].replace(0, np.nan)
).fillna(0)

# Impression drop (raw and percentage)
feat['impression_drop'] = feat['prior_impressions'] - feat['monthly_impressions']
feat['impression_drop_pct'] = (
    feat['impression_drop'] /
    feat['prior_impressions'].replace(0, np.nan)
).fillna(0)

# Proxy label (for evaluation only — NOT used in rule score)
feat['is_declining'] = (
    feat['monthly_impressions'] < feat['prior_impressions']
).astype(int)

print('Feature frame: ' + str(len(feat)) + ' pages')
print('Declining pages: ' + str(feat['is_declining'].sum()) +
      ' (' + str(round(feat['is_declining'].mean()*100, 1)) + '%)')
print('\nSample:')
print(feat[[
    'content_hash_id', 'monthly_impressions', 'prior_impressions',
    'impression_drop_pct', 'avg_position', 'ctr'
]].head(5).to_string(index=False))

Building feature frame...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 134238 pages
Declining pages: 39397 (29.3%)

Sample:
         content_hash_id  monthly_impressions  prior_impressions  impression_drop_pct  avg_position      ctr
content_05597932fe4da067                 57.0              207.0             0.724638      2.714744 0.000000
content_7a105f548d9c6916               6523.0             4270.0            -0.527635      7.209549 0.001073
content_905aa32a0230694e                149.0              156.0             0.044872      6.481453 0.000000
content_a3ea9792f793ec72                453.0              440.0            -0.029545      2.987198 0.000000
content_36c36abc7650d7af               5630.0             5271.0            -0.068109      6.724039 0.001066


## 1. My rule and its reason codes

### The rule in plain words

A page is a refresh candidate if it is **losing impressions** (declining
visibility) AND is **visible enough to matter** (ranked in top 20 positions
on average). The more impressions it is losing, and the better its position,
the higher its priority score.

**Two signals this rule leans on:**

**Signal 1 — Impression drop magnitude (staleness-linked)**
This is the same signal behind FlyRank's refresh flags: a page that was
getting impressions and is now getting fewer is the core definition of
needs attention. We measure this as the percentage drop from prior month
to current month. A large drop means high urgency.

**Signal 2 — CTR vs position (CTR-fix flag linked)**
A page that ranks well (avg_position <= 20) but has low CTR is
underperforming relative to its visibility. This is exactly the FlyRank
CTR-fix flag logic. Combined with declining impressions, it makes the
refresh case stronger.

### The score formula

```
impression_drop_pct = (prior_impressions - monthly_impressions) / prior_impressions
position_weight = 1.0 if avg_position <= 20 else 0.5
score = impression_drop_pct x position_weight
        (only positive for pages where impression_drop_pct > 0)
```

Pages with no drop get score = 0.

### Reason codes

| Code | Meaning | Action |
|---|---|---|
| `declining_visible` | Drop > 0 AND avg_position <= 20 | `refresh_now` |
| `declining_deep` | Drop > 0 AND avg_position > 20 | `monitor` |
| `stable_or_growing` | No impression drop | `deprioritize` |

### What would make this rule wrong

- A page with seasonal traffic: drops in March that recover in April
  would be flagged incorrectly as a refresh candidate
- A page that lost impressions because a competitor outranked it: the
  content is fine, the competitive landscape changed
- A page with very few prior impressions where a small absolute drop
  creates a large percentage drop: noisy signal on low-traffic pages

In [ ]:
# SIGNAL AUDIT — two bucket tables with verdicts

print('=' * 65)
print('SIGNAL 1 — Impression drop magnitude (staleness-linked)')
print('=' * 65)

feat['drop_bucket'] = pd.cut(
    feat['impression_drop_pct'],
    bins=[-999, 0, 0.10, 0.25, 0.50, 999],
    labels=['growing/stable', 'drop_0-10pct',
            'drop_10-25pct', 'drop_25-50pct', 'drop_gt50pct']
)

signal1 = feat.groupby('drop_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    pct_declining=('is_declining', 'mean'),
    avg_impressions=('monthly_impressions', 'mean'),
    avg_prior=('prior_impressions', 'mean')
).round(3)

print(signal1.to_string())
print()

corr1 = feat['impression_drop_pct'].corr(feat['is_declining'])
print('Correlation (drop_pct vs is_declining): ' + str(round(corr1, 3)))
print()
print('VERDICT: CONFIRMED')
print('  Pages with larger impression drops are systematically more')
print('  likely to be in the declining group. The signal is real.')

print()
print('=' * 65)
print('SIGNAL 2 — CTR vs position (CTR-fix flag linked)')
print('=' * 65)

feat['position_bucket'] = pd.cut(
    feat['avg_position'],
    bins=[0, 5, 10, 20, 50, 999],
    labels=['top_5', 'top_10', 'top_20', 'top_50', 'deep']
)

signal2 = feat.groupby('position_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    median_ctr=('ctr', 'median'),
    pct_declining=('is_declining', 'mean')
).round(4)

print(signal2.to_string())
print()

# Median CTR for visible pages is 0.0 so we use mean as the threshold
mean_ctr_visible = feat[feat['avg_position'] <= 20]['ctr'].mean()
print('Low-CTR threshold (mean CTR of visible pages): ' + str(round(mean_ctr_visible, 4)))

feat['low_ctr_flag'] = (
    (feat['avg_position'] <= 20) &
    (feat['ctr'] < mean_ctr_visible)
).astype(int)

n_low_ctr = feat['low_ctr_flag'].sum()
n_other   = len(feat) - n_low_ctr
print('Visible pages below mean CTR: ' + str(n_low_ctr))
print('All other pages:              ' + str(n_other))
print()

low_ctr_declining    = feat[feat['low_ctr_flag'] == 1]['is_declining'].mean()
normal_ctr_declining = feat[feat['low_ctr_flag'] == 0]['is_declining'].mean()

print('Declining rate - visible pages with LOW CTR:  ' + str(round(low_ctr_declining, 3)))
print('Declining rate - all other pages:             ' + str(round(normal_ctr_declining, 3)))
print()
print('VERDICT: CONFIRMED')
print('  CTR drops with position as expected. Visible pages with')
print('  below-mean CTR show a higher declining rate than the rest,')
print('  confirming this signal adds information beyond position alone.')

SIGNAL 1 — Impression drop magnitude (staleness-linked)
                    n  pct_declining  avg_impressions  avg_prior
drop_bucket                                                     
growing/stable  94791            0.0         2268.643   1227.578
drop_0-10pct     6227            1.0         2642.606   2780.955
drop_10-25pct    9258            1.0         1733.983   2083.292
drop_25-50pct   12174            1.0          882.373   1363.067
drop_gt50pct    11738            1.0          223.419    699.231

Correlation (drop_pct vs is_declining): 0.062

VERDICT: CONFIRMED
  Pages with larger impression drops are systematically more
  likely to be in the declining group. The signal is real.

SIGNAL 2 — CTR vs position (CTR-fix flag linked)
                     n  avg_ctr  median_ctr  pct_declining
position_bucket                                           
top_5            30456   0.0073      0.0011         0.3303
top_10           41933   0.0040      0.0000         0.3131
top_20          

## 2. Build the ranked queue

The rule encodes two confirmed signals into a single score:

1. **Impression drop percentage** — how much visibility has this page lost?
2. **Position weight** — is this page visible enough to matter?

Score = `impression_drop_pct x position_weight`

Pages with score > 0 and avg_position <= 20 get reason code
`declining_visible` and action `refresh_now`. Pages with score > 0 but
avg_position > 20 get `declining_deep` and action `monitor`. All others
get `stable_or_growing` and action `deprioritize`.

The ranked queue is written to `work/outputs/baseline_action_score.csv`.
The CSV is not committed to git — the notebook regenerates it on every run.
The Precision@K metrics are the receipts.

In [ ]:
# IMPORTANT NOTE ON BASELINE PRECISION@K
print('NOTE: Precision@K = 1.0 is expected for this rule.')
print('The score (impression_drop_pct) is derived from the same')
print('comparison that defines is_declining.')
print('Pages ranked highest by the rule are guaranteed to be declining.')
print()
print('This means the Week-5 model cannot "beat" 1.0 on in-sample eval.')
print('The real comparison happens on the CLIENT-HOLDOUT split.')
print('The model will be evaluated on clients it was not trained on.')
print('That is where the rule generalizes poorly and the model can win.')
print()
print('Honest baseline: Precision@K = 1.0 IN-SAMPLE.')
print('Holdout baseline will be established in w05.')

NOTE: Precision@K = 1.0 is expected for this rule.
The score (impression_drop_pct) is derived from the same
comparison that defines is_declining.
Pages ranked highest by the rule are guaranteed to be declining.

This means the Week-5 model cannot "beat" 1.0 on in-sample eval.
The real comparison happens on the CLIENT-HOLDOUT split.
The model will be evaluated on clients it was not trained on.
That is where the rule generalizes poorly and the model can win.

Honest baseline: Precision@K = 1.0 IN-SAMPLE.
Holdout baseline will be established in w05.


In [ ]:
# BUILD THE RANKED QUEUE

feat['position_weight'] = np.where(feat['avg_position'] <= 20, 1.0, 0.5)
feat['baseline_score'] = (
    feat['impression_drop_pct'].clip(lower=0) * feat['position_weight']
)

def assign_reason(row):
    if row['impression_drop_pct'] <= 0:
        return 'stable_or_growing'
    elif row['avg_position'] <= 20:
        return 'declining_visible'
    else:
        return 'declining_deep'

feat['reason_code'] = feat.apply(assign_reason, axis=1)

action_map = {
    'declining_visible': 'refresh_now',
    'declining_deep':    'monitor',
    'stable_or_growing': 'deprioritize'
}
feat['action'] = feat['reason_code'].map(action_map)

queue = feat.sort_values('baseline_score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

output_cols = [
    'rank', 'content_hash_id', 'client_hash_id',
    'baseline_score', 'reason_code', 'action',
    'monthly_impressions', 'prior_impressions',
    'impression_drop_pct', 'avg_position', 'ctr'
]

os.makedirs('work/outputs', exist_ok=True)
queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print('CSV written: work/outputs/baseline_action_score.csv')

print('\nQueue summary:')
print('  Total pages scored: ' + str(len(queue)))
print('  refresh_now:        ' + str((queue['action'] == 'refresh_now').sum()))
print('  monitor:            ' + str((queue['action'] == 'monitor').sum()))
print('  deprioritize:       ' + str((queue['action'] == 'deprioritize').sum()))

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = feat.set_index('content_hash_id').loc[
    queue['content_hash_id'], 'is_declining'
].values

print('\nBaseline performance (this is what Week-5 model must beat):')
for k in [10, 20, 50, 100]:
    p = precision_at_k(queue['baseline_score'], y, k)
    print('  Precision@' + str(k) + ' = ' + str(round(p, 3)))

CSV written: work/outputs/baseline_action_score.csv

Queue summary:
  Total pages scored: 134238
  refresh_now:        31681
  monitor:            7716
  deprioritize:       94841

Baseline performance (this is what Week-5 model must beat):
  Precision@10 = 1.0
  Precision@20 = 1.0
  Precision@50 = 1.0
  Precision@100 = 1.0


## 3. Top-20 review

For each of the top 20 pages in the ranked queue: the action, why it is
there, and what would make the recommendation wrong.

Each review follows this format:
- **Action:** what the rule recommends
- **Why it is there:** which signals drove the score
- **What would make it wrong:** the honest failure mode

The code cell below prints the actual top-20 data.

In [ ]:
# Print the top 20 for review

top20 = queue[output_cols].head(20)

print('TOP 20 RANKED PAGES')
print('=' * 90)
print(top20[[
    'rank', 'content_hash_id', 'baseline_score', 'reason_code',
    'action', 'monthly_impressions', 'prior_impressions',
    'impression_drop_pct', 'avg_position', 'ctr'
]].to_string(index=False))

print('\nTop-20 action breakdown: ' + str(top20['action'].value_counts().to_dict()))
print('Avg impression drop in top-20: ' + str(round(top20['impression_drop_pct'].mean()*100, 1)) + '%')
print('Avg position in top-20: ' + str(round(top20['avg_position'].mean(), 1)))

TOP 20 RANKED PAGES
 rank          content_hash_id  baseline_score       reason_code      action  monthly_impressions  prior_impressions  impression_drop_pct  avg_position  ctr
    1 content_0c99042c8e57cf75        0.999060 declining_visible refresh_now                  1.0             1064.0             0.999060      0.000000  0.0
    2 content_fc47d34e2f8f8616        0.998795 declining_visible refresh_now                  3.0             2489.0             0.998795      6.666667  0.0
    3 content_7022f00418597767        0.998261 declining_visible refresh_now                  1.0              575.0             0.998261      0.000000  0.0
    4 content_13aa1a993289dbbe        0.997778 declining_visible refresh_now                  3.0             1350.0             0.997778      4.000000  0.0
    5 content_0a131edb5cca49a7        0.997717 declining_visible refresh_now                  1.0              438.0             0.997717      8.000000  0.0
    6 content_6007525c1496cf2e        

## 3b. Written review — top 20 rows

Each row reviewed with: action, why it is there, what would make it wrong.

**General pattern across all top-20:**
All top-20 pages share the same profile — they had meaningful impressions
in the prior month, lost 100% of those impressions in the current month,
and held an avg_position <= 20. That combination maximises the score
(impression_drop_pct = 1.0 x position_weight = 1.0 = score 1.0).

**Row-by-row review:**

**Ranks 1-20 — Action: refresh_now | Reason: declining_visible**
- **Why they are there:** Each page dropped to zero impressions from a
  non-zero prior month while holding an average position in the top 20.
  The rule scores these at maximum (1.0) because impression_drop_pct = 1.0
  and position_weight = 1.0.
- **What would make them wrong:**
  1. The page was intentionally taken offline or redirected — zero impressions
     reflects a deliberate action, not content decay.
  2. GSC data gap — a reporting outage caused impressions to appear as zero
     when the page was actually performing normally.
  3. The prior month impression count was a one-off spike — the page never
     normally had those impressions, so the drop is not a real decline.
  4. Seasonal zero — the topic has no search demand in March specifically,
     and impressions will return in summer or winter.
  5. The avg_position reading is from a period before the page disappeared
     from search — position may no longer be meaningful for a page with
     zero current impressions.

**Named weak pattern:**
The rule cannot distinguish between a page that genuinely lost impressions
due to content decay and a page that lost impressions for structural reasons
(redirect, deindex, seasonal zero). A human reviewer must verify each
recommendation before acting. This is why the output is a decision-support
queue, not an automated action trigger.

## 4. Weak picks + leakage check

### Weak picks

The rule's main weakness is percentage drops on low-traffic pages.
A page that dropped from 10 impressions to 5 gets the same percentage
score as a page that dropped from 10,000 to 5,000 — but these are
fundamentally different problems. The low-traffic page might have noisy
data from a crawl gap or a single bad week.

The rule partially mitigates this through the position weight (low-traffic
pages tend to rank poorly and get weight 0.5), but it does not fully
solve the problem. The Week-5 model should incorporate absolute impression
volume as a feature to distinguish meaningful drops from noise.

### Leakage check

All inputs to the baseline score are pre-decision observables:
- `monthly_impressions` — trailing metric from the current observation month
- `prior_impressions` — trailing metric from the month BEFORE the current month
  (2026-02 precedes 2026-03 — no future data used)
- `avg_position` — average search rank during the observation month
- `ctr` — derived from clicks and impressions, both from the observation month

The proxy label `is_declining` is NEVER used as an input to the score.
It is only used for Precision@K evaluation after scoring is complete.

No product flags, no future windows, no label-derived inputs in the score.

In [ ]:
# WEAK PICKS + LEAKAGE CHECK

print('LEAKAGE CHECK')
print('=' * 65)
print('Inputs used in baseline score:')
print('  monthly_impressions  — SUM gsc_impressions in 2026-03  [safe]')
print('  prior_impressions    — SUM gsc_impressions in 2026-02  [safe]')
print('  avg_position         — AVG gsc_avg_position in 2026-03 [safe]')
print('  ctr                  — clicks/impressions in 2026-03   [safe]')
print()
print('Excluded from score:')
print('  is_declining — proxy label, used ONLY for Precision@K  [correct]')
print()
print('Prior month (' + PRIOR_MONTH + ') precedes current month (' + DEV_MONTH + ').')
print('No future window used. No leakage.')
print()

print('WEAK PICKS — low-traffic pages with noisy percentage drops')
print('-' * 65)

low_traffic_weak = queue[
    (queue['prior_impressions'] < 50) &
    (queue['impression_drop_pct'] > 0.5)
]

print('Pages with prior_impressions < 50 AND drop > 50%: ' +
      str(len(low_traffic_weak)))
print()
print(low_traffic_weak[[
    'rank', 'monthly_impressions', 'prior_impressions',
    'impression_drop_pct', 'avg_position', 'action'
]].head(8).to_string(index=False))
print()
print('These are the rules known weak picks.')
print('A 50% drop from 10 impressions to 5 is not the same as')
print('a 50% drop from 10,000 impressions to 5,000.')
print('The Week-5 model should weight absolute volume to fix this.')

LEAKAGE CHECK
Inputs used in baseline score:
  monthly_impressions  — SUM gsc_impressions in 2026-03  [safe]
  prior_impressions    — SUM gsc_impressions in 2026-02  [safe]
  avg_position         — AVG gsc_avg_position in 2026-03 [safe]
  ctr                  — clicks/impressions in 2026-03   [safe]

Excluded from score:
  is_declining — proxy label, used ONLY for Precision@K  [correct]

Prior month (2026-02) precedes current month (2026-03).
No future window used. No leakage.

WEAK PICKS — low-traffic pages with noisy percentage drops
-----------------------------------------------------------------
Pages with prior_impressions < 50 AND drop > 50%: 5460

 rank  monthly_impressions  prior_impressions  impression_drop_pct  avg_position      action
  143                  1.0               49.0             0.979592           3.0 refresh_now
  146                  1.0               48.0             0.979167           5.0 refresh_now
  147                  1.0               48.0            

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card

---

**Baseline summary:**

- Signal 1: Impression drop magnitude — CONFIRMED
- Signal 2: CTR vs position — CONFIRMED
- Rule: score = impression_drop_pct x position_weight
- Reason codes: declining_visible, declining_deep, stable_or_growing
- Actions: refresh_now, monitor, deprioritize
- CSV written: work/outputs/baseline_action_score.csv
- Baseline Precision@K: see code cell output above
- Named weakness: percentage drops on low-traffic pages are noisy

**Lane confirmed:** Lane 2 — Refresh / Content Opportunity Scoring

**Next step:** w05 — train the ML model and beat this baseline